In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2" #set to -1 to use CPU
import torch
from diffusers import AutoencoderKLWan
from pipelines.pipeline_wan_i2v import WanImageToVideoPipeline
from transformers import CLIPVisionModel
from diffusers.utils import export_to_video, export_to_gif
from PIL import Image
from IPython import display
import numpy as np
import decord

model_id = "Wan-AI/Wan2.1-I2V-14B-480P-Diffusers"
image_encoder = CLIPVisionModel.from_pretrained(model_id, subfolder="image_encoder", torch_dtype=torch.float32)
# vae = AutoencoderKLWan.from_pretrained(model_id, subfolder="vae", torch_dtype=torch.float32)
vae = AutoencoderKLWan.from_pretrained(model_id, subfolder="vae", torch_dtype=torch.bfloat16)
pipe = WanImageToVideoPipeline.from_pretrained(model_id, vae=vae, image_encoder=image_encoder, torch_dtype=torch.bfloat16)
# pipe.enable_sequential_cpu_offload()
pipe = pipe.to("cuda")

pipe.transformer.enable_gradient_checkpointing()
# pipe.vae.enable_tiling()
# pipe.vae.enable_slicing()
# pipe.vae.enable_gradient_checkpointing()


In [ ]:
# video load
def make_frames(path):
    frames = []
    vr = decord.VideoReader(path)
    for i in range(len(vr)):
        frames.append(Image.fromarray(vr[i].asnumpy()))
    return frames

# parse frame numbers
def parse_line_to_list(line: str) -> list:
    """
    parse_line_to_list("1,3,4") -> [1, 3, 4]
    parse_line_to_list("1-3,4") -> [1, 2, 3, 4]
    """
    result = []
    parts = line.split(',')
    
    for part in parts:
        part = part.strip()
        if '-' in part:
            start_str, end_str = part.split('-')
            start, end = int(start_str), int(end_str)
            result.extend(range(start, end + 1))
        else:
            result.append(int(part))
    
    return result

## Keyframe-guided generation

In [ ]:
init = Image.open("examples/turtle_wan/0.png")
last = Image.open("examples/turtle_wan/48.png")

video_init = [None] * 81
video_init[:41] = [init] * 41
video_init[41:] = [last] * 40

In [ ]:
seed = 0
st = 1.0

prompt = "A mature sea turtle glides through clear blue waters above a coral reef, its flippers moving gracefully. Sunlight filters through, casting a tranquil glow on the turtle and its serene surroundings."
negative_prompt = "low quality"

guidance_lr = [3e0] * 5 + [3e0] * 45
guidance_step = [5]* 5 + [3] * 5 + [1] * 10 + [0] * 30 # small step also allows keyframe guidance

travel_time = (15, 20) # in paper we use (5, 20), but applying time-travel in later timesteps (ex: (7, 20)) for better details (early time-travel causes detail drop)
fixed_frames = "40,80" # 48 means the last frame
fixed_frames_ = parse_line_to_list(fixed_frames) # makes it a list of frame numbers

save_dir = "results/turtle_wan.gif"

video = pipe(
    prompt=prompt,
    video=video_init,
    negative_prompt=negative_prompt, 
    num_videos_per_prompt=1,
    num_inference_steps=50,
    guidance_scale=5.0,
    generator=torch.Generator(device="cuda").manual_seed(seed),
    fixed_frames=fixed_frames_,
    guidance_step=guidance_step,
    guidance_lr=guidance_lr,
    loss_fn="frame",
    height=480, 
    width=832, 
    num_frames=81, 
    latent_downscale_factor=4
).frames[0]

output_ = (video * 255).astype(np.uint8)
output_ = [Image.fromarray(frame) for frame in output_]

export_to_gif(output_, save_dir, fps=16)
display.Image(filename=save_dir)